In [13]:
from google.colab import userdata
from google import genai
from google.genai import types
from typing import TypedDict, List, Optional
import json
import re
import time

#Secret api key
API_KEY = userdata.get("GEN_AI")
client = genai.Client(api_key=API_KEY)

#Model
MODEL_NAME = "gemini-2.5-flash"

#Schema
class Education(TypedDict):
    Degree: Optional[str]
    Institute: Optional[str]
    Year: Optional[str]
    Cgpa: Optional[str]

class Experience(TypedDict):
    Company: Optional[str]
    Role: Optional[str]
    Duration: Optional[str]
    Highlights: List[str]

class Project(TypedDict):
    Project: Optional[str]
    Description: Optional[str]
    Tech_stack: List[str]

class Resume(TypedDict):
    Name: Optional[str]
    Email: Optional[str]
    Phone: Optional[str]
    Linkedin_url: Optional[str]
    Education: List[Education]
    Skills: List[str]
    Experience: List[Experience]
    Projects: List[Project]
    Total_experience_years: float
    Summary: Optional[str]

#SystemInstructions
SYSTEM_INSTRUCTION = """
You are an expert HR resume parser.

Rules:
1. Extract only information present in the resume.
2. Do not invent values.
3. Use null for missing fields.
4. Output must be valid JSON only.
5. Skills should be clean and unique.
6. Normalize phone numbers to +91XXXXXXXXXX if possible.
7. Summary must be one sentence.
8. Resume can be in English or Any Indian languages.
9. Output values must always be in English.
"""

RESPONSE = {
    "type": "OBJECT",
    "properties": {
        "Name": {"type": "STRING"},
        "Email": {"type": "STRING"},
        "Phone": {"type": "STRING"},
        "Linkedin_url": {"type": "STRING"},
        "Education": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "Name of the degree": {"type": "STRING"},
                    "Name of the institute": {"type": "STRING"},
                    "Year of study": {"type": "STRING"},
                    "Cgpa": {"type": "STRING"}
                }
            }
        },
        "Skills": {"type": "ARRAY", "items": {"type": "STRING"}},
        "Experience": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "Name of the company": {"type": "STRING"},
                    "Role": {"type": "STRING"},
                    "Duration": {"type": "STRING"},
                    "Highlights": {"type": "ARRAY", "items": {"type": "STRING"}}
                }
            }
        },
        "Projects": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "Name": {"type": "STRING"},
                    "Description": {"type": "STRING"},
                    "Tech_stack": {"type": "ARRAY", "items": {"type": "STRING"}}
                }
            }
        },
        "Total_experience_years": {"type": "NUMBER"},
        "Summary": {"type": "STRING"}
    }
}

def validate_email(email):
    if email is None: return True
    return "@" in email

def validate_year(year):
    if year is None: return True
    match = re.findall(r"\\d{4}", str(year))
    if not match: return True
    year_num = int(match[0])
    return 1990 <= year_num <= 2026

def validate_experience(exp):
    try: return float(exp) >= 0
    except: return False

def parse_resume(text: str) -> dict:
    prompt = f"Parse this resume and return JSON.\\n\\nResume:\\n{text}"
    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_INSTRUCTION,
                    response_mime_type="application/json",
                    response_schema=RESPONSE,
                    temperature=0.2,
                    max_output_tokens=2048
                )
            )
            data = json.loads(response.text)
            if not validate_email(data.get("email")): data["email"] = None
            for edu in data.get("education", []):
                if not validate_year(edu.get("year")): edu["year"] = None
            if not validate_experience(data.get("total_experience_years", 0)):
                data["total_experience_years"] = 0.0
            return data
        except Exception as e:
            print(f"Retry {attempt + 1}... Error: {e}")
            time.sleep(20)
    return {"error": "Model unavailable"}

def parse_prompt_only(text):
    response = client.models.generate_content(model=MODEL_NAME, contents=f"Extract resume info as JSON:\\n{text}")
    return response.text

def parse_mime_type(text):
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=f"Extract resume info as JSON:\\n{text}",
        config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0.2)
    )
    return response.text

def parse_schema(text):
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=f"Extract resume info:\\n{text}",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            response_mime_type="application/json",
            response_schema=RESPONSE,
            temperature=0.2
        )
    )
    return response.text

#TEST
resume_text = """
Name: Rohit Sharma
Email: Rohit45@gmail.com
Phone: 9826443264
Skills: Python, Java, A.I
Experience: Fresher
Projects: Data analytics with Tableau
"""

try:
    print("Test-1 Prompt Only")
    print(parse_prompt_only(resume_text))
    time.sleep(15)

    print("\\nTest-2 — MimeType")
    print(parse_mime_type(resume_text))
    time.sleep(15)

    print("\\nTest-3 — Response schema")
    print(parse_schema(resume_text))
    time.sleep(15)

    print("\\nFinal Parsed Output")
    print(json.dumps(parse_resume(resume_text), indent=2))
except Exception as e:
    print(f"\\nAn error occurred during testing: {e}")
    print("Please wait 60 seconds and run the cell again to reset your quota.")

Test-1 Prompt Only
```json
{
  "name": "Rohit Sharma",
  "email": "Rohit45@gmail.com",
  "phone": "9826443264",
  "skills": [
    "Python",
    "Java",
    "A.I"
  ],
  "experience": "Fresher",
  "projects": [
    {
      "title": "Data analytics with Tableau"
    }
  ]
}
```
\nTest-2 — MimeType
{
  "name": "Rohit Sharma",
  "email": "Rohit45@gmail.com",
  "phone": "9826443264",
  "skills": [
    "Python",
    "Java",
    "A.I"
  ],
  "experience": "Fresher",
  "projects": [
    "Data analytics with Tableau"
  ]
}
\nTest-3 — Response schema
{"Name": "Rohit Sharma", "Email": "Rohit45@gmail.com", "Phone": "+919826443264", "Linkedin_url": "null", "Education": [], "Skills": ["Python", "Java", "A.I"], "Experience": [], "Projects": [{"Name": "Data analytics with Tableau", "Description": "null", "Tech_stack": ["Tableau"]}], "Total_experience_years": 0, "Summary": "null"}
\nFinal Parsed Output
{
  "Name": "Rohit Sharma",
  "Email": "Rohit45@gmail.com",
  "Phone": "+919826443264",
  "Linkedin_u